# NQX-Core vs TurboQuant
## Comparison of KV-cache quantization methods

- **NautilusQuant (NQX)**: Structured golden-angle Givens rotations (deterministic, no PRNG state)
- **TurboQuant**: Random orthogonal matrix (requires PRNG + full matrix storage)

In [ ]:
import numpy as np

from nqx.constants import NQXConfig
from nqx.cpu import NQXCore
import demos.turboquant_emul as tq

In [ ]:
cfg = NQXConfig(dim=128, bits=3)
nq_core = NQXCore(cfg)

rng = np.random.default_rng(0)
x = rng.standard_normal((1024, 128)).astype(np.float32)

# NQX encode/decode
enc_nq = nq_core.encode(x)
dec_nq = nq_core.decode(enc_nq)
rmse_nq = float(np.sqrt(((x - dec_nq.reconstructed) ** 2).mean()))

# TurboQuant encode/decode
enc_tq = tq.encode(x, bits=3, qjl_alpha=0.5, seed=0)
dec_tq = tq.decode(enc_tq)
rmse_tq = tq.rmse(x, dec_tq)

# Compression
ratio_nq = x.size * 2 / len(enc_nq.packed_bytes)
ratio_tq = x.size * 2 / (enc_tq.q.nbytes + enc_tq.sign.nbytes + enc_tq.mins.nbytes + enc_tq.maxs.nbytes)

print(f"{'Metric':<25} {'NQX':>12} {'TurboQuant':>12}")
print(f"{'':-<25} {'':->12} {'':->12}")
print(f"{'RMSE':<25} {rmse_nq:>12.4f} {rmse_tq:>12.4f}")
print(f"{'Compression ratio':<25} {ratio_nq:>12.2f}x {ratio_tq:>12.2f}x")
print(f"{'State size (matrix)':<25} {'0 (ROM)':>12} {tq.state_size_bytes(cfg):>12,} bytes")

In [ ]:
# Energy comparison
nq_energy = nq_core.energy.total_nj()
tq_energy_dict = tq.encode_energy_pj(cfg, 1024)
tq_energy = tq_energy_dict['total_nj']

print(f"{'Energy (total)':<25} {nq_energy:>12.1f} nJ {tq_energy:>12.1f} nJ")
print(f"{'Energy per vector':<25} {nq_energy/1024:>12.3f} nJ {tq_energy/1024:>12.3f} nJ")

In [ ]:
# Bar chart comparison
try:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    
    names = ['NQX', 'TurboQuant']
    rmses = [rmse_nq, rmse_tq]
    ratios = [ratio_nq, ratio_tq]
    energies = [nq_energy/1024, tq_energy/1024]
    
    axes[0].bar(names, rmses, color=['#d4af37', '#4a90d9'])
    axes[0].set_title('RMSE')
    axes[0].set_ylabel('RMSE')
    
    axes[1].bar(names, ratios, color=['#d4af37', '#4a90d9'])
    axes[1].set_title('Compression ratio')
    axes[1].set_ylabel('x')
    
    axes[2].bar(names, energies, color=['#d4af37', '#4a90d9'])
    axes[2].set_title('Energy per vector')
    axes[2].set_ylabel('nJ')
    
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not available — ASCII table above")